In [6]:
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

train_set =pd.read_csv('train.csv')
test_set = pd.read_csv('test.csv')

X = train_set.drop(columns=["id", "target"]).values.astype(np.float32)
y = train_set["target"].values


In [ ]:
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1, 0.01, 0.05],
    'n_estimators': [50, 100, 200],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Create XGBClassifier
model = XGBClassifier(objective='multi:softmax', num_class=3, random_state=42, n_jobs=1)

# Same CV as the NN notebook (10 folds x 3 repeats on all 105 rows), so the scores are comparable
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=0)

# Perform grid search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=cv, n_jobs=-1)
grid_search.fit(X, y)

# Print best score and parameters
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
print(f"Best parameters: {grid_search.best_params_}")

# Best model (GridSearchCV already refit it on all 105 rows)
best_model = grid_search.best_estimator_

Best CV accuracy: 0.924
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}


In [8]:
# Predict on the test set (best_model is already trained on all 105 training rows)
X_test = test_set.drop(columns=["id"]).values.astype(np.float32)
test_pred = best_model.predict(X_test)

submission = pd.DataFrame({"id": test_set["id"], "target": test_pred})
submission.to_csv("submission_xgboost.csv", index=False)
print(submission["target"].value_counts().sort_index())

target
0    19
1    13
2    13
Name: count, dtype: int64
